# Lab 12 · Filter an orientation predicate

**Time:** 90 minutes, followed by capstone work. **Preparation:** [Lecture 12](../notebooks/12_robust_geometry.ipynb). Implement the precision filter and exact fallback, and keep uncertain measured inputs separate.

In [1]:
from fractions import Fraction
from vc.intervals import RationalInterval as Interval
from vc.geometry import determinant, exact_point, orientation_bound, sign_of_interval, OrientationResult

## Task A · Filter and fall back (45 minutes)

Implement `student_orientation` for exact rational coordinates. Convert with `exact_point`, form singleton intervals, try the configured precisions, and return an `OrientationResult`. If all attempted bounds are inconclusive, evaluate the determinant using exact fractions. Use `sign_of_interval`: zero is conclusive only for the singleton [0,0].

In [ ]:
def student_orientation(a, b, c, precisions = (24, 53, 100, 200)):
    raise NotImplementedError("Preserve exact inputs, try enclosing arithmetic, then fall back")

In [ ]:
n = 2 ** 27
points = ((0, 0), (n, n - 1), (n + 1, n))
assert determinant(*[tuple(float(x) for x in point) for point in points]) == 0
result = student_orientation(*points)
assert result.sign == 1
fallback = student_orientation(*points, precisions = (24,))
assert fallback.method == "fraction" and fallback.exact_determinant == 1
assert student_orientation(points[0], points[2], points[1]).sign == -1
assert student_orientation((0, 0), ("1/3", "2/3"), ("2/3", "4/3")).sign == 0
for precision, bound in result.attempts:
    print(precision, bound)
print("Final sign:", result.sign)

**Written response:** Prove the sign filter and explain why exact fallback settles collinearity for rational inputs. Does the test establish the correctness of an entire geometry algorithm?

## Task B · State the intended inputs (20 minutes)

Compare exact decimal input with the exact value of a stored float. The examples below intentionally specify two different mathematical triples.

In [ ]:
decimal = student_orientation((0, 0), (1, "0.1"), (10, 1))
stored_tenth = Fraction.from_float(0.1)
stored = student_orientation((0, 0), (1, stored_tenth), (10, 1))
print("Intended decimal:", decimal.sign)
print("Stored binary tenth:", stored.sign)
assert decimal.sign == 0
assert stored.sign == -1

**Written response:** Why are both results correct? Which part of the certificate must say whether a decimal or stored binary input is intended?

## Task C · Keep physical uncertainty (25 minutes)

Enclose orientation for measured coordinates below at 24, 100, and 200 bits. Give two allowed exact realizations with opposite signs.

In [ ]:
a = (Interval(0), Interval(0))
b = (Interval(1), Interval(1))
c = (Interval(2), Interval("1.999", "2.001"))
for precision in [24, 100, 200]:
    bound = orientation_bound(a, b, c, precision)
    print(precision, bound, sign_of_interval(bound))
    assert sign_of_interval(bound) is None
negative = student_orientation((0, 0), (1, 1), (2, "1.999"))
positive = student_orientation((0, 0), (1, 1), (2, "2.001"))
assert negative.sign == -1 and positive.sign == 1

**Written response:** Why is midpoint fallback invalid for this family? State an accurate final conclusion, then restart the kernel and run the completed lab.